# **STEPS**



1.   Document loader - from lanchain document loader
2.   Text Splitter
3.   Embedding
4.   Vector Stores
5.   Retriver
6.   Chains




In [ ]:
!pip install -qU langchain-google-genai langchain-community langchain chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.8/68.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 76.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 82.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.4/236.4 kB 16.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.

In [ ]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 18.2 MB/s eta 0:00:00


In [ ]:
# Step 1 : Document loader
from langchain_community.document_loaders import TextLoader # It allow us to read txt files,

# ✅ NEW — use this for PDFs
from langchain_community.document_loaders import PyPDFLoader

/tmp/ipykernel_1023/2493986948.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader # It allow us to read txt files,


In [ ]:
#step 2 : Text spilitter
#chunking part

# from langchain text splitter
from langchain_text_splitters import RecursiveCharacterTextSplitter # Recusive test splitter would work good in text splitting


In [ ]:
#step 3: Embeddings:
# Text to Vectors(To  numbers)

from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI

In [ ]:
# 4. VEctore Store
# is the place where the vectors will be stored after embedding
# Eg: Chroma DB, Faiis, Pine corn etc..
# used for similarity searchs

from langchain_community.vectorstores import Chroma

In [ ]:
from langchain_core.prompts import ChatPromptTemplate #allow us to create prompts
from langchain_core.output_parsers import StrOutputParser # allow us to parse the output

In [ ]:
#5. Retrivers


# This is going to find the relivent document for a question from the chunks which is stored in the vector DB
# Retrivers works on similarity search

In [ ]:
# 6.Chains

# Connect everything -> user Question - Retriver - Documents(Chunks) - LLMs - Final answer


# Compaining all of these we can create RAG

from langchain_core.runnables import RunnablePassthrough # this will help us to create a chain

In [ ]:
from google.colab import userdata
API = userdata.get('Key1')


In [ ]:
def load_and_split(filepath):
 loader = PyPDFLoader(filepath)

 docs = loader.load() #page wise data

 print("data splitting to chunks")
 splitter = RecursiveCharacterTextSplitter(chunk_size = 500,chunk_overlap = 150)
 splits = splitter.split_documents(docs)
 print(f'split {len(splits)}chunks')
 return splits

In [ ]:
from decimal import Context
def create_rag_chain(splits):
  embeddings = GoogleGenerativeAIEmbeddings(model = 'gemini-embedding-001',task_type= 'RETRIEVAL_DOCUMENT', api_key=API)
  vector_store = Chroma.from_documents(documents = splits,embedding = embeddings)
  retriever = vector_store.as_retriever()# help data from document

  llm = ChatGoogleGenerativeAI(model = 'gemini-2.5-flash',temperature = 0.2, api_key=API)

  #System prompt
  template = """Answer the question as truthfully as possible using the the provided context : {context}, and if the answer is not contained within the text below, say "I don't know"
    Question = {question}
    Helpful Answer:"""

  #we will create doc content pages
  def format_docs(docs):
    return '\n\n'.join(doc.page_content for doc in docs)

  prompt = ChatPromptTemplate.from_template(template) # Define prompt here

  # Rag Chain
  chain = (
      {
          'context' : retriever | format_docs,
          'question' : RunnablePassthrough()
      }
      | prompt
      | llm
      | StrOutputParser()
      #'ChatPromptTemplate.from_template(template)' # what is this this was suggested by the colab in the place of "|prompt"
  )

  return chain

In [ ]:
 file_path = "/content/AMAL.pdf"

In [ ]:
splits = load_and_split(file_path)
rag_chain = create_rag_chain(splits)

data splitting to chunks
split 14chunks


In [ ]:
message = input("Enter your question: ")
output = rag_chain.invoke(message)
print(output)

Enter your question: Who is amal john?
Amal John is a Data Analyst with experience in data reporting, dashboard development, and business intelligence solutions. They are skilled in Advanced Excel (Pivot Tables, Power Query, formulas), Power BI, SQL, and Python, and experienced in data collection, validation, cleaning, and transformation. They hold a Master of Computer Sciences (Data Analytics) from Rajagiri College of Social Sciences and a Bachelor of Science in Statistics from St. Thomas College Autonomous.


In [ ]:
# Gradio

In [ ]:
import gradio as gr

In [ ]:
from logging import PlaceHolder
def respond(message, history):
  return rag_chain.invoke(message)

demo = gr.ChatInterface(
    fn = respond,
    textbox = gr.Textbox(placeholder= "Hey Charlieee....! How'u doing...")
)

demo.launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c68964cbd82cfb545b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
